# LeetCode 725 — Split Linked List in Parts bằng Python

Link đề: https://leetcode.com/problems/split-linked-list-in-parts/ · Độ khó: **Medium** · Chủ đề: Linked List

Skeleton LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def splitListToParts(self, head: Optional[ListNode], k: int) -> List[Optional[ListNode]]:
        ...
```

Ghi chú: bài rèn hai kỹ năng nền của danh sách liên kết: **đếm độ dài** rồi **cắt đúng chỗ** (`node.next = None`) mà không làm mất phần còn lại. Phần "chia đều" chỉ là phép chia lấy nguyên và phần dư (`divmod`). Vì kết quả là các **nút**, ta tự dựng `ListNode`, hàm dựng danh sách từ list Python và hàm xuất ngược lại để kiểm thử.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chia danh sách liên kết đơn thành đúng `k` đoạn liên tiếp, độ dài chênh nhau tối đa 1, đoạn đứng trước không ngắn hơn đoạn đứng sau; đoạn thiếu nút là `None`.
2. Nắm các công cụ Python: lớp `ListNode`, duyệt `while cur:`, phép `//`, `%` và `divmod`, cắt liên kết `prev.next = None`, danh sách chứa `None`.
3. Đi từ **chép sang list rồi dựng lại** → **đếm + cắt tại chỗ** → **mảng nút + chỉ số** → **chuyển nút sang đoạn mới bằng dummy**.
4. Hiểu mấu chốt: với `n` nút và `k` đoạn, mỗi đoạn có `n // k` nút và **`n % k` đoạn đầu** được thêm 1 nút; khi cắt phải **giữ lại `next` trước khi đặt `next = None`**.
5. Tránh các lỗi kinh điển: quên cắt (các đoạn dính nhau), đưa phần dư cho các đoạn cuối, `AttributeError` khi `k > n` (đi tiếp trên `None`), làm mất phần còn lại do cắt trước khi lưu `next`.

## 2. Đề bài & ví dụ

Cho `head` là đầu của một danh sách liên kết đơn và số nguyên `k`, hãy chia danh sách thành `k` đoạn **liên tiếp**.

- Độ dài các đoạn phải **gần bằng nhau nhất có thể**: hai đoạn bất kỳ hơn kém nhau không quá 1 nút. Có thể có đoạn rỗng (`None`).
- Các đoạn giữ **thứ tự xuất hiện** trong danh sách; đoạn đứng trước luôn có độ dài **lớn hơn hoặc bằng** đoạn đứng sau.

Trả về mảng gồm `k` đoạn.

**Ví dụ 1**

- Input: `head = [1,2,3]`, `k = 5`
- Output: `[[1],[2],[3],[],[]]`
- Giải thích: phần tử đầu `output[0]` có `val = 1`, `next = null`; hai phần tử cuối là `null` (biểu diễn là `[]`).

**Ví dụ 2**

- Input: `head = [1,2,3,4,5,6,7,8,9,10]`, `k = 3`
- Output: `[[1,2,3,4],[5,6,7],[8,9,10]]`

**Ràng buộc:**

- Số nút trong danh sách nằm trong khoảng `[0, 1000]`
- `0 <= Node.val <= 1000`
- `1 <= k <= 50`

Hệ quả: danh sách có thể **rỗng** (`head = None`), và `k` có thể **lớn hơn** số nút (nhiều đoạn rỗng). Số nút nhỏ nên một lời giải `O(n + k)` là quá đủ.

In [1]:
from typing import List, Optional
import random


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

    def __repr__(self):
        return f"ListNode({self.val})"


def build_list(vals):
    """Dựng danh sách liên kết từ list Python; list rỗng -> None."""
    dummy = ListNode()
    tail = dummy
    for v in vals:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head, limit=100000):
    """Xuất danh sách liên kết ra list Python (có giới hạn để không lặp vô hạn nếu có chu trình)."""
    out = []
    while head is not None and len(out) < limit:
        out.append(head.val)
        head = head.next
    return out


ex1 = ([1, 2, 3], 5, [[1], [2], [3], [], []])
ex2 = (list(range(1, 11)), 3, [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]])
for vals, k, expected in (ex1, ex2):
    head = build_list(vals)
    print("head =", to_list(head), "| k =", k, "| kỳ vọng:", expected)
print("danh sách rỗng ->", build_list([]), to_list(build_list([])))

head = [1, 2, 3] | k = 5 | kỳ vọng: [[1], [2], [3], [], []]
head = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] | k = 3 | kỳ vọng: [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]
danh sách rỗng -> None []


## 3. Kiến thức nền

### 3.1. Danh sách liên kết đơn và duyệt `while cur:`

Mỗi nút (`ListNode`) có `val` và con trỏ `next` trỏ tới nút kế tiếp; nút cuối có `next = None`. Ta chỉ giữ **đầu** `head`. Để đi qua mọi nút ta dùng con trỏ chạy `cur`:

```
cur = head
while cur:        # cur khác None thì còn nút
    ... dùng cur.val ...
    cur = cur.next
```

In [2]:
head = build_list([4, 5, 6])
cur = head
while cur:
    print("nút", cur.val, "-> next là", cur.next)
    cur = cur.next
print("head vẫn là", head, "(ta chỉ dịch con trỏ cur, không đổi head)")

nút 4 -> next là ListNode(5)
nút 5 -> next là ListNode(6)
nút 6 -> next là None
head vẫn là ListNode(4) (ta chỉ dịch con trỏ cur, không đổi head)


### 3.2. Đếm độ dài

Muốn chia đều ta phải biết tổng số nút `n`. Duyệt một lần, tăng bộ đếm. Danh sách rỗng (`head = None`) cho `n = 0` vì vòng `while` không chạy lần nào.

In [3]:
def length(head):
    n = 0
    while head:
        n += 1
        head = head.next
    return n


print(length(build_list([4, 5, 6])), length(build_list([])), length(build_list(range(1000))))

3 0 1000


### 3.3. `//`, `%` và `divmod`

- `a // b`: thương nguyên (làm tròn xuống). `a % b`: phần dư.
- `divmod(a, b)` trả luôn cặp `(a // b, a % b)`.

Chia `n = 10` nút cho `k = 3` đoạn: `divmod(10, 3) = (3, 1)` — mỗi đoạn có 3 nút và còn dư 1 nút phải chia cho 1 đoạn (đoạn đầu). Chia `n = 3` cho `k = 5`: `divmod(3, 5) = (0, 3)` — mỗi đoạn 0 nút, 3 đoạn đầu được thêm 1 nút.

In [4]:
for n, k in [(10, 3), (3, 5), (12, 4), (0, 7), (7, 1)]:
    q, r = divmod(n, k)
    sizes = [q + (1 if i < r else 0) for i in range(k)]
    print(f"n={n:2d}, k={k}: q={q}, r={r} -> kích thước các đoạn {sizes}, tổng = {sum(sizes)}")

n=10, k=3: q=3, r=1 -> kích thước các đoạn [4, 3, 3], tổng = 10
n= 3, k=5: q=0, r=3 -> kích thước các đoạn [1, 1, 1, 0, 0], tổng = 3
n=12, k=4: q=3, r=0 -> kích thước các đoạn [3, 3, 3, 3], tổng = 12
n= 0, k=7: q=0, r=0 -> kích thước các đoạn [0, 0, 0, 0, 0, 0, 0], tổng = 0
n= 7, k=1: q=7, r=0 -> kích thước các đoạn [7], tổng = 7


### 3.4. Cắt liên kết và quy tắc "lưu `next` trước khi cắt"

Để tách một đoạn ra khỏi danh sách, ta đặt `tail.next = None` ở nút cuối của đoạn. Nhưng **ngay sau khi đặt, ta mất đường tới phần còn lại**. Vì thế phải lưu trước:

```
nxt = tail.next      # 1. lưu phần còn lại
tail.next = None     # 2. cắt
cur = nxt            # 3. tiếp tục từ phần còn lại
```

In [5]:
head = build_list([1, 2, 3, 4, 5])
tail = head.next                 # nút 2
nxt = tail.next                  # lưu phần còn lại: 3 -> 4 -> 5
tail.next = None                 # cắt: đoạn đầu chỉ còn 1 -> 2
print("đoạn đầu:", to_list(head), "| phần còn lại:", to_list(nxt))

đoạn đầu: [1, 2] | phần còn lại: [3, 4, 5]


### 3.5. `list` chứa `None`, và slicing

`[None] * 3` tạo list 3 phần tử `None`. Kết quả của bài là list các đầu-đoạn, nơi đoạn rỗng chính là `None`. Slicing `vals[i:i + size]` lấy đoạn `size` phần tử bắt đầu từ `i` (không lỗi khi vượt biên) — hữu ích cho cách vét cạn.

In [6]:
print([None] * 3)
vals = [1, 2, 3, 4, 5]
print(vals[0:2], vals[2:5], vals[5:5], vals[3:99])

[None, None, None]
[1, 2] [3, 4, 5] [] [4, 5]


## 4. Ý tưởng nền tảng / trực giác

Hãy làm tay với ví dụ 2: `n = 10` nút, `k = 3` đoạn. Nếu chia đều tuyệt đối thì mỗi đoạn `10 / 3 = 3,33` nút — không thể. Mỗi đoạn có ít nhất `10 // 3 = 3` nút, dư `10 % 3 = 1` nút; nút dư đó cho đoạn **đầu** (đề yêu cầu đoạn trước dài hơn hoặc bằng đoạn sau). Vậy kích thước `[4, 3, 3]`:

```
1 2 3 4 | 5 6 7 | 8 9 10
 đoạn 0   đoạn 1   đoạn 2
```

Ví dụ 1: `n = 3`, `k = 5`: `3 // 5 = 0`, dư 3 → ba đoạn đầu một nút, hai đoạn cuối **rỗng**: `[1] [2] [3] [] []`.

Một cách nghĩ khác: **chia bài**. Phát lần lượt từng nút cho đoạn `0, 1, ..., k-1, 0, 1, ...` (vòng tròn); đoạn nào nhận trước thì nhiều hơn hoặc bằng. Số nút đoạn `i` nhận là số `t` trong `0..n-1` có `t % k == i`.

In [7]:
def sizes_by_dealing(n, k):
    """Chia bài vòng tròn: đoạn i nhận nút t nếu t % k == i."""
    sizes = [0] * k
    for t in range(n):
        sizes[t % k] += 1
    return sizes


def sizes_by_divmod(n, k):
    q, r = divmod(n, k)
    return [q + (1 if i < r else 0) for i in range(k)]


for n, k in [(10, 3), (3, 5), (0, 4), (50, 50), (1000, 50), (999, 7)]:
    assert sizes_by_dealing(n, k) == sizes_by_divmod(n, k)
    print(f"n={n:4d}, k={k:2d}: {sizes_by_divmod(n, k)[:8]}{'...' if k > 8 else ''}")

n=  10, k= 3: [4, 3, 3]
n=   3, k= 5: [1, 1, 1, 0, 0]
n=   0, k= 4: [0, 0, 0, 0]
n=  50, k=50: [1, 1, 1, 1, 1, 1, 1, 1]...
n=1000, k=50: [20, 20, 20, 20, 20, 20, 20, 20]...
n= 999, k= 7: [143, 143, 143, 143, 143, 142, 142]


## 5. Cách 1 — Brute Force: chép sang list, cắt lát, dựng lại

### Ý tưởng

1. Chép mọi giá trị ra list Python `vals` (`to_list`).
2. Tính `q, r = divmod(n, k)`.
3. Với mỗi đoạn `i`, độ dài `size = q + (1 if i < r else 0)`; lấy `vals[start:start + size]` rồi **dựng danh sách mới** bằng `build_list`.

Rất khó sai nên ta dùng cách này làm **oracle** tham chiếu. Nhược điểm: tạo nút mới (không dùng lại nút gốc) và tốn thêm bộ nhớ.

### Code

In [8]:
def split_via_list(head, k):
    vals = to_list(head)
    q, r = divmod(len(vals), k)
    parts = []
    start = 0
    for i in range(k):
        size = q + (1 if i < r else 0)
        parts.append(build_list(vals[start:start + size]))
        start += size
    return parts


for vals, k, expected in (ex1, ex2):
    parts = split_via_list(build_list(vals), k)
    print([to_list(p) for p in parts], "==", expected, [to_list(p) for p in parts] == expected)

[[1], [2], [3], [], []] == [[1], [2], [3], [], []] True
[[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]] == [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]] True


### Độ phức tạp và nút thắt

- Thời gian `O(n + k)`, bộ nhớ thêm `O(n)` (list chép + các nút mới).
- Nút thắt: ta **không cần** chép; chỉ cần đi dọc danh sách và **cắt** đúng chỗ. Cắt tại chỗ dùng bộ nhớ thêm `O(1)` (ngoài mảng kết quả `k` phần tử).

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát:

1. **Kích thước từng đoạn biết trước** nếu biết `n`: `size_i = n // k + (1 if i < n % k else 0)`. Vậy một lượt duyệt để đếm `n`.
2. Mỗi đoạn là một dãy nút liên tiếp. Với kích thước `size`, ta từ đầu đoạn đi `size - 1` bước để đến **nút cuối đoạn**, rồi cắt: lưu `nxt = tail.next`, đặt `tail.next = None`, và đoạn kế tiếp bắt đầu từ `nxt`.

Vậy thuật toán gồm hai lượt: **lượt 1** đếm `n`; **lượt 2** duyệt, ở mỗi đoạn đi `size - 1` bước rồi cắt. Tổng bước đi lượt 2 là `n`, nên toàn bộ `O(n + k)`. Trường hợp `size = 0` (đoạn rỗng): đoạn là `None`, không đi bước nào và không cắt.

In [9]:
# Minh họa nhịp "đi size - 1 bước, lưu next, cắt" trên ví dụ 2
head = build_list(range(1, 11))
n = length(head)
q, r = divmod(n, 3)
cur = head
for i in range(3):
    size = q + (1 if i < r else 0)
    part_head = cur
    for _ in range(size - 1):
        cur = cur.next                   # đi tới nút cuối của đoạn
    nxt = cur.next                       # lưu phần còn lại
    cur.next = None                      # cắt
    cur = nxt
    print(f"đoạn {i}: size={size}, nội dung {to_list(part_head)}, phần còn lại {to_list(cur)}")

đoạn 0: size=4, nội dung [1, 2, 3, 4], phần còn lại [5, 6, 7, 8, 9, 10]
đoạn 1: size=3, nội dung [5, 6, 7], phần còn lại [8, 9, 10]
đoạn 2: size=3, nội dung [8, 9, 10], phần còn lại []


## 7. Các cách tiếp theo

Tổng cộng 4 cách: (1) chép sang list rồi dựng lại (mục 5), (2) đếm + cắt tại chỗ, (3) mảng nút + chỉ số, (4) chuyển từng nút sang đoạn mới bằng nút `dummy`.

### 7.1. Cách 2 — Đếm + cắt tại chỗ (lời giải nộp)

Giải thích từng đoạn:

1. Lượt 1: đếm `n`; `q, r = divmod(n, k)`.
2. Vòng `for i in range(k)`: `part_head = cur` là đầu đoạn `i`; `size = q + (i < r)`.
3. Nếu `size > 0`: đi `size - 1` bước để `cur` là nút cuối; lưu `nxt = cur.next`; `cur.next = None`; `cur = nxt`.
4. Nếu `size = 0`: `cur` lúc này là `None` (mọi nút đã dùng hết) nên `part_head` là `None` — đúng đoạn rỗng.
5. Thêm `part_head` vào kết quả.

In [10]:
def split_count_cut(head, k):
    n = 0
    cur = head
    while cur:
        n += 1
        cur = cur.next
    q, r = divmod(n, k)
    parts = []
    cur = head
    for i in range(k):
        part_head = cur
        size = q + (1 if i < r else 0)
        if size > 0:
            for _ in range(size - 1):
                cur = cur.next
            nxt = cur.next
            cur.next = None
            cur = nxt
        parts.append(part_head)
    return parts


for vals, k, expected in (ex1, ex2):
    parts = split_count_cut(build_list(vals), k)
    print([to_list(p) for p in parts] == expected, [to_list(p) for p in parts])

True [[1], [2], [3], [], []]
True [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]


In [11]:
# Code nộp LeetCode cho cách 2 (đếm + cắt tại chỗ)
class SolutionCountCut:
    def splitListToParts(self, head: Optional[ListNode], k: int) -> List[Optional[ListNode]]:
        n = 0
        cur = head
        while cur:
            n += 1
            cur = cur.next
        q, r = divmod(n, k)
        ans = [None] * k
        cur = head
        for i in range(k):
            ans[i] = cur
            size = q + (1 if i < r else 0)
            if size:
                for _ in range(size - 1):
                    cur = cur.next
                nxt = cur.next
                cur.next = None
                cur = nxt
        return ans


print([to_list(p) for p in SolutionCountCut().splitListToParts(build_list(ex2[0]), 3)])

[[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]


**Độ phức tạp:** lượt đếm `O(n)`, lượt cắt đi qua mỗi nút đúng một lần `O(n)`, cộng `k` lần ghi kết quả → `O(n + k)` thời gian; bộ nhớ thêm `O(1)` ngoài mảng kết quả `k` phần tử. Các đoạn **dùng lại nút gốc** (không tạo nút mới).

### 7.2. Cách 3 — Mảng nút + chỉ số

Thay vì đi bước từng đoạn, ta chép **các tham chiếu nút** vào một list `nodes` (không chép giá trị), sau đó tính **chỉ số bắt đầu** của đoạn `i` bằng công thức:

```
start_i = i * q + min(i, r)        # i đoạn trước có tổng q*i + (số đoạn trước được +1, tối đa r)
size_i  = q + (1 if i < r else 0)
```

Đoạn `i` (nếu `size_i > 0`) có đầu `nodes[start_i]`, nút cuối `nodes[start_i + size_i - 1]` và ta đặt `next = None` cho nút cuối đó. Nhờ truy cập ngẫu nhiên nên không cần "lưu `next` trước khi cắt" — cắt theo chỉ số không làm mất nút nào vì mọi nút đã nằm sẵn trong mảng.

In [12]:
def split_node_array(head, k):
    nodes = []
    cur = head
    while cur:
        nodes.append(cur)
        cur = cur.next
    q, r = divmod(len(nodes), k)
    parts = []
    for i in range(k):
        size = q + (1 if i < r else 0)
        if size == 0:
            parts.append(None)
            continue
        start = i * q + min(i, r)
        nodes[start + size - 1].next = None
        parts.append(nodes[start])
    return parts


for vals, k, expected in (ex1, ex2):
    parts = split_node_array(build_list(vals), k)
    print([to_list(p) for p in parts] == expected, [to_list(p) for p in parts])

# kiểm tra công thức start_i
q, r = divmod(10, 3)
print([i * q + min(i, r) for i in range(3)])      # [0, 4, 7]

True [[1], [2], [3], [], []]
True [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]
[0, 4, 7]


In [13]:
# Code nộp LeetCode cho cách 3
class SolutionNodeArray:
    def splitListToParts(self, head: Optional[ListNode], k: int) -> List[Optional[ListNode]]:
        nodes = []
        cur = head
        while cur:
            nodes.append(cur)
            cur = cur.next
        q, r = divmod(len(nodes), k)
        ans = []
        for i in range(k):
            size = q + (1 if i < r else 0)
            if size == 0:
                ans.append(None)
                continue
            start = i * q + min(i, r)
            nodes[start + size - 1].next = None
            ans.append(nodes[start])
        return ans


print([to_list(p) for p in SolutionNodeArray().splitListToParts(build_list(ex1[0]), 5)])

[[1], [2], [3], [], []]


**Độ phức tạp:** `O(n + k)` thời gian, `O(n)` bộ nhớ thêm cho mảng `nodes` (tham chiếu). Đổi lại, code gọn và dễ kiểm tra chỉ số.

### 7.3. Cách 4 — Chuyển từng nút sang đoạn mới bằng `dummy`

Cách nghĩ khác: không "cắt" mà **tháo từng nút ra khỏi danh sách gốc và nối vào đuôi đoạn mới**. Với mỗi đoạn tạo một nút giả `dummy` làm gốc tạm và con trỏ `tail`; lặp `size` lần: lấy `node = cur`, tiến `cur = cur.next` (**lưu trước**), đặt `node.next = None`, nối `tail.next = node`, rồi `tail = node`. Đầu đoạn là `dummy.next` (là `None` nếu `size = 0`, đúng yêu cầu đoạn rỗng).

In [14]:
def split_dummy_relink(head, k):
    n = 0
    cur = head
    while cur:
        n += 1
        cur = cur.next
    q, r = divmod(n, k)
    parts = []
    cur = head
    for i in range(k):
        size = q + (1 if i < r else 0)
        dummy = ListNode()
        tail = dummy
        for _ in range(size):
            node = cur
            cur = cur.next           # lưu phần còn lại TRƯỚC khi sửa node.next
            node.next = None
            tail.next = node
            tail = node
        parts.append(dummy.next)
    return parts


for vals, k, expected in (ex1, ex2):
    parts = split_dummy_relink(build_list(vals), k)
    print([to_list(p) for p in parts] == expected, [to_list(p) for p in parts])

True [[1], [2], [3], [], []]
True [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]


In [15]:
# Code nộp LeetCode cho cách 4
class SolutionDummy:
    def splitListToParts(self, head: Optional[ListNode], k: int) -> List[Optional[ListNode]]:
        n = 0
        cur = head
        while cur:
            n += 1
            cur = cur.next
        q, r = divmod(n, k)
        ans = []
        cur = head
        for i in range(k):
            dummy = ListNode()
            tail = dummy
            for _ in range(q + (1 if i < r else 0)):
                node = cur
                cur = cur.next
                node.next = None
                tail.next = node
                tail = node
            ans.append(dummy.next)
        return ans


print([to_list(p) for p in SolutionDummy().splitListToParts(build_list(ex2[0]), 3)])

[[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]


**Độ phức tạp:** `O(n + k)` thời gian, `O(k)` nút `dummy` tạm (mỗi đoạn một nút, tổng `O(k)`), không mảng phụ theo `n`. Hơi dài hơn cách 2 nhưng **không bao giờ gọi `.next` trên `None`** (vòng lặp chỉ chạy đúng `size` lần).

## 8. TRACE từng bước

### 8.1. Ví dụ 2: `head = [1..10]`, `k = 3`

Lượt 1: `n = 10`, `q = 3`, `r = 1`.

| Đoạn `i` | `size` | `part_head` | đi `size-1` bước tới | `nxt = cur.next` | sau cắt, đoạn | còn lại |
|---|---|---|---|---|---|---|
| 0 | 4 (vì `0 < 1`) | 1 | 4 | 5 | `1,2,3,4` | `5..10` |
| 1 | 3 | 5 | 7 | 8 | `5,6,7` | `8,9,10` |
| 2 | 3 | 8 | 10 | `None` | `8,9,10` | rỗng |

### 8.2. Ví dụ 1: `head = [1,2,3]`, `k = 5`

`n = 3`, `q = 0`, `r = 3`: kích thước `[1, 1, 1, 0, 0]`.

| Đoạn `i` | `size` | Hành động | Đoạn |
|---|---|---|---|
| 0 | 1 | đi 0 bước, cắt sau nút 1 | `[1]` |
| 1 | 1 | cắt sau nút 2 | `[2]` |
| 2 | 1 | cắt sau nút 3 (`nxt = None`) | `[3]` |
| 3 | 0 | `part_head = cur = None` | `[]` |
| 4 | 0 | `None` | `[]` |

Hàm `trace_split` in lại đúng các bước này.

In [16]:
def trace_split(head, k):
    print("danh sách:", to_list(head), "| k =", k)
    n = length(head)
    q, r = divmod(n, k)
    print(f"n = {n}, q = n // k = {q}, r = n % k = {r}")
    cur = head
    parts = []
    for i in range(k):
        size = q + (1 if i < r else 0)
        part_head = cur
        if size > 0:
            steps = 0
            for _ in range(size - 1):
                cur = cur.next
                steps += 1
            nxt = cur.next
            tail_val = cur.val
            cur.next = None
            cur = nxt
            print(f"đoạn {i}: size={size}, đi {steps} bước, nút cuối = {tail_val}, "
                  f"đoạn = {to_list(part_head)}, phần còn lại = {to_list(cur)}")
        else:
            print(f"đoạn {i}: size=0 -> None (cur = {cur})")
        parts.append(part_head)
    return parts


trace_split(build_list(range(1, 11)), 3)
print()
trace_split(build_list([1, 2, 3]), 5)

danh sách: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] | k = 3
n = 10, q = n // k = 3, r = n % k = 1
đoạn 0: size=4, đi 3 bước, nút cuối = 4, đoạn = [1, 2, 3, 4], phần còn lại = [5, 6, 7, 8, 9, 10]
đoạn 1: size=3, đi 2 bước, nút cuối = 7, đoạn = [5, 6, 7], phần còn lại = [8, 9, 10]
đoạn 2: size=3, đi 2 bước, nút cuối = 10, đoạn = [8, 9, 10], phần còn lại = []

danh sách: [1, 2, 3] | k = 5
n = 3, q = n // k = 0, r = n % k = 3
đoạn 0: size=1, đi 0 bước, nút cuối = 1, đoạn = [1], phần còn lại = [2, 3]
đoạn 1: size=1, đi 0 bước, nút cuối = 2, đoạn = [2], phần còn lại = [3]
đoạn 2: size=1, đi 0 bước, nút cuối = 3, đoạn = [3], phần còn lại = []
đoạn 3: size=0 -> None (cur = None)
đoạn 4: size=0 -> None (cur = None)


[ListNode(1), ListNode(2), ListNode(3), None, None]

### 8.3. Trace cách 4 (`dummy`): trạng thái từng nút trên ví dụ nhỏ

Với `head = [1,2,3,4]`, `k = 3`: `n = 4`, `q = 1`, `r = 1` → kích thước `[2, 1, 1]`. Cell dưới in từng lần chuyển nút.

In [17]:
def trace_dummy(head, k):
    n = length(head)
    q, r = divmod(n, k)
    cur = head
    for i in range(k):
        size = q + (1 if i < r else 0)
        dummy = ListNode()
        tail = dummy
        print(f"--- đoạn {i}, size = {size} ---")
        for _ in range(size):
            node = cur
            cur = cur.next
            node.next = None
            tail.next = node
            tail = node
            print(f"  chuyển nút {node.val}: đoạn đang là {to_list(dummy.next)}, gốc còn {to_list(cur)}")
        print("  đoạn xong:", to_list(dummy.next))


trace_dummy(build_list([1, 2, 3, 4]), 3)

--- đoạn 0, size = 2 ---
  chuyển nút 1: đoạn đang là [1], gốc còn [2, 3, 4]
  chuyển nút 2: đoạn đang là [1, 2], gốc còn [3, 4]
  đoạn xong: [1, 2]
--- đoạn 1, size = 1 ---
  chuyển nút 3: đoạn đang là [3], gốc còn [4]
  đoạn xong: [3]
--- đoạn 2, size = 1 ---
  chuyển nút 4: đoạn đang là [4], gốc còn []
  đoạn xong: [4]


## 9. Code nộp LeetCode

Chọn cách 2 (đếm + cắt tại chỗ): ngắn, `O(n + k)`, `O(1)` bộ nhớ thêm.

In [18]:
class Solution:
    def splitListToParts(self, head: Optional[ListNode], k: int) -> List[Optional[ListNode]]:
        n = 0
        cur = head
        while cur:
            n += 1
            cur = cur.next
        q, r = divmod(n, k)           # mỗi đoạn có q nút, r đoạn đầu được thêm 1
        ans = [None] * k
        cur = head
        for i in range(k):
            ans[i] = cur
            size = q + (1 if i < r else 0)
            if size:
                for _ in range(size - 1):
                    cur = cur.next    # đi tới nút cuối của đoạn
                nxt = cur.next        # lưu phần còn lại
                cur.next = None       # cắt
                cur = nxt
        return ans


sol = Solution()
print([to_list(p) for p in sol.splitListToParts(build_list([1, 2, 3]), 5)])
print([to_list(p) for p in sol.splitListToParts(build_list(range(1, 11)), 3)])

[[1], [2], [3], [], []]
[[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]


## 10. Vì sao đúng

**Kích thước.** Gọi `n` là số nút. Đặt `q = n // k`, `r = n % k`, `size_i = q + [i < r]`. Khi đó `sum(size_i) = k*q + r = n` (đủ nút), hai đoạn bất kỳ hơn kém nhau tối đa 1 (chỉ nhận giá trị `q` hoặc `q + 1`), và các đoạn `q + 1` đứng **trước** các đoạn `q` (vì chúng là `r` đoạn đầu). Vậy kích thước thỏa đúng ba điều kiện của đề.

**Tính duy nhất của kích thước.** Nếu hai đoạn chênh tối đa 1 và tổng là `n` thì có đúng `r` đoạn kích thước `q + 1` và `k - r` đoạn kích thước `q`; yêu cầu "đoạn trước ≥ đoạn sau" cố định vị trí của chúng như trên. Nên lời giải là **duy nhất**.

**Cắt đúng.** Bất biến vòng lặp: trước khi xử lý đoạn `i`, `cur` là đầu của phần danh sách **chưa** chia (gồm đúng `n - (size_0 + ... + size_{i-1})` nút) và các đoạn `0..i-1` đã được tách rời hẳn (đuôi mỗi đoạn có `next = None`).
- Nếu `size_i > 0`: đi `size_i - 1` bước để `cur` là nút thứ `size_i` của phần chưa chia (tồn tại vì còn đủ nút); lưu `nxt`, đặt `cur.next = None` để đoạn `i` đúng `size_i` nút; `cur = nxt` giữ bất biến.
- Nếu `size_i = 0`: khi đó `i >= r` và `q = 0`, tức `n < k`; các đoạn trước đã dùng hết `n` nút nên `cur = None`, và `part_head = None` là đoạn rỗng đúng.

**Không gọi `.next` trên `None`.** Chỉ đi bước khi `size_i > 0` và còn đủ `size_i` nút, nên các lần `cur = cur.next` và `cur.next` đều hợp lệ.

In [19]:
# Kiểm chứng công thức kích thước với định nghĩa trực tiếp của đề, mọi (n, k) trong miền nhỏ
for n in range(0, 60):
    for k in range(1, 20):
        sizes = sizes_by_divmod(n, k)
        assert sum(sizes) == n
        assert max(sizes) - min(sizes) <= 1
        assert sizes == sorted(sizes, reverse=True)
        assert sizes == sizes_by_dealing(n, k)
print("công thức kích thước thỏa 3 điều kiện của đề với n = 0..59, k = 1..19")

công thức kích thước thỏa 3 điều kiện của đề với n = 0..59, k = 1..19


## 11. Time / Space Complexity

| Cách | Time | Space (thêm) |
|---|---|---|
| 1. Chép sang list, dựng lại | `O(n + k)` | `O(n)` (list + nút mới) |
| 2. Đếm + cắt tại chỗ (nộp) | `O(n + k)` | `O(1)` ngoài mảng kết quả |
| 3. Mảng nút + chỉ số | `O(n + k)` | `O(n)` |
| 4. Chuyển nút sang đoạn mới | `O(n + k)` | `O(k)` nút `dummy` |

Mảng kết quả luôn `O(k)`. Cell dưới đếm số lần "đi bước" `cur = cur.next` của cách 2 để thấy tổng đúng bằng số nút (mỗi nút được bước qua một lần ở lượt 2, trừ nút cuối của mỗi đoạn được xử lý bởi `nxt = cur.next`).

In [20]:
def count_moves(n, k):
    head = build_list(range(n))
    q, r = divmod(n, k)
    cur, moves = head, 0
    for i in range(k):
        size = q + (1 if i < r else 0)
        if size:
            for _ in range(size - 1):
                cur = cur.next
                moves += 1
            cur = cur.next          # bước qua nút cuối của đoạn (qua nxt)
            moves += 1
    return moves


for n, k in [(10, 3), (3, 5), (1000, 50), (1000, 1), (0, 7)]:
    print(f"n={n:4d}, k={k:2d}: số bước = {count_moves(n, k):4d}")
    assert count_moves(n, k) == n

n=  10, k= 3: số bước =   10
n=   3, k= 5: số bước =    3
n=1000, k=50: số bước = 1000
n=1000, k= 1: số bước = 1000
n=   0, k= 7: số bước =    0


## 12. Bộ test

Chiến lược: (a) ví dụ đề; (b) edge case (danh sách rỗng, một nút, `k = 1`, `k > n`, `k = n`, kích thước tối đa); (c) phản ví dụ cho các cách sai; (d) đối chiếu **mọi cách** với một oracle độc lập (chia bài vòng tròn) trên nhiều input ngẫu nhiên có seed.

Vì hàm sửa danh sách tại chỗ, mỗi lần chạy ta **dựng lại danh sách mới**. Hàm `check_parts` kiểm các điều sau: có đúng `k` đoạn; kích thước khớp oracle chia bài; ghép các đoạn lại được danh sách gốc; mọi đoạn kết thúc đúng (không dính đoạn khác); đoạn rỗng là `None`; (với cách tại chỗ) các nút đều là **nút gốc**; và không nút nào xuất hiện hai lần.

In [21]:
def check_parts(vals, k, parts, in_place=True, nodes_before=None):
    n = len(vals)
    assert isinstance(parts, list) and len(parts) == k, "phải có đúng k đoạn"
    lists = [to_list(p, limit=5000) for p in parts]
    assert [len(x) for x in lists] == sizes_by_dealing(n, k), ("kích thước sai", [len(x) for x in lists])
    flat = [v for part in lists for v in part]
    assert flat == vals, "ghép lại không ra danh sách gốc"
    for p, lst in zip(parts, lists):
        if not lst:
            assert p is None, "đoạn rỗng phải là None"
    # không nút nào xuất hiện ở hai đoạn
    seen = set()
    for p in parts:
        cur = p
        while cur is not None:
            assert id(cur) not in seen, "nút xuất hiện hai lần"
            seen.add(id(cur))
            cur = cur.next
    if in_place and nodes_before is not None:
        assert seen == nodes_before, "đoạn phải dùng lại đúng các nút gốc"


def nodes_of(head):
    out = set()
    while head is not None:
        out.add(id(head))
        head = head.next
    return out


solvers = {
    "chép list": (split_via_list, False),
    "đếm+cắt": (split_count_cut, True),
    "mảng nút": (split_node_array, True),
    "dummy": (split_dummy_relink, True),
    "nộp CountCut": (lambda h, k: SolutionCountCut().splitListToParts(h, k), True),
    "nộp NodeArray": (lambda h, k: SolutionNodeArray().splitListToParts(h, k), True),
    "nộp Dummy": (lambda h, k: SolutionDummy().splitListToParts(h, k), True),
    "nộp Solution": (lambda h, k: Solution().splitListToParts(h, k), True),
}


def check_all(vals, k, label=""):
    for name, (f, in_place) in solvers.items():
        head = build_list(vals)
        before = nodes_of(head)
        parts = f(head, k)
        try:
            check_parts(vals, k, parts, in_place, before)
        except AssertionError as e:
            raise AssertionError((label, name, vals, k, str(e)))


# (a) ví dụ đề: so với kết quả đã cho
for vals, k, expected in (ex1, ex2):
    for name, (f, _) in solvers.items():
        got = [to_list(p) for p in f(build_list(vals), k)]
        assert got == expected, (name, got)
    check_all(vals, k, "ví dụ")
print("ví dụ đề: OK")

ví dụ đề: OK


In [22]:
# (b) edge case
check_all([], 1, "rỗng, k=1")
check_all([], 7, "rỗng, k=7")
check_all([5], 1, "một nút, k=1")
check_all([5], 50, "một nút, k=50")
check_all([1, 2, 3, 4], 1, "k=1: một đoạn là cả danh sách")
check_all([1, 2, 3, 4], 4, "k=n")
check_all([1, 2, 3, 4], 3, "n=4, k=3")
check_all(list(range(1000)), 50, "n=1000, k=50")
check_all(list(range(1000)), 1, "n=1000, k=1")
check_all(list(range(999)), 50, "n=999, k=50 (dư 49)")
check_all([1000] * 7, 7, "giá trị biên 1000, k=n")
check_all([0] * 5, 50, "toàn số 0, k > n")

# kiểm tra trực tiếp vài kết quả mong đợi
assert [to_list(p) for p in Solution().splitListToParts(build_list([]), 3)] == [[], [], []]
assert [to_list(p) for p in Solution().splitListToParts(build_list([1, 2, 3, 4]), 3)] == [[1, 2], [3], [4]]
print("edge case: OK")

edge case: OK


In [23]:
# (c) phản ví dụ: những cách sai (phân tích ở mục 13)
def wrong_no_cut(head, k):
    """SAI: đi đúng số bước nhưng không đặt next = None."""
    n = length(head)
    q, r = divmod(n, k)
    parts, cur = [], head
    for i in range(k):
        parts.append(cur)
        size = q + (1 if i < r else 0)
        for _ in range(size):
            if cur:
                cur = cur.next
    return parts


def wrong_extra_last(head, k):
    """SAI: phần dư chia cho các đoạn CUỐI thay vì các đoạn đầu."""
    n = length(head)
    q, r = divmod(n, k)
    parts, cur = [], head
    for i in range(k):
        size = q + (1 if i >= k - r else 0)
        part_head = cur if size else None
        if size:
            for _ in range(size - 1):
                cur = cur.next
            nxt = cur.next
            cur.next = None
            cur = nxt
        parts.append(part_head)
    return parts


def wrong_crash_k_gt_n(head, k):
    """SAI: không kiểm tra size == 0, đi .next trên None."""
    n = length(head)
    q, r = divmod(n, k)
    parts, cur = [], head
    for i in range(k):
        size = q + (1 if i < r else 0)
        part_head = cur
        for _ in range(size - 1):
            cur = cur.next
        nxt = cur.next            # cur có thể là None khi size == 0
        cur.next = None
        cur = nxt
        parts.append(part_head)
    return parts


def wrong_lose_rest(head, k):
    """SAI: đặt next = None TRƯỚC khi lưu next."""
    n = length(head)
    q, r = divmod(n, k)
    parts, cur = [], head
    for i in range(k):
        size = q + (1 if i < r else 0)
        part_head = cur
        if size and cur:
            for _ in range(size - 1):
                cur = cur.next
            cur.next = None
            cur = cur.next        # SAI: cur.next đã bị xóa -> None
        parts.append(part_head)
    return parts


def wrong_ceil_each(head, k):
    """SAI: mọi đoạn đều lấy ceil(n / k) nút cho tới khi hết."""
    n = length(head)
    size = -(-n // k)
    parts, cur = [], head
    for i in range(k):
        part_head = cur
        if cur:
            for _ in range(size - 1):
                if cur.next:
                    cur = cur.next
            nxt = cur.next
            cur.next = None
            cur = nxt
        parts.append(part_head)
    return parts


for f in (wrong_no_cut, wrong_extra_last, wrong_crash_k_gt_n, wrong_lose_rest, wrong_ceil_each):
    try:
        got = [to_list(p, limit=50) for p in f(build_list(ex2[0]), 3)]
        got2 = [to_list(p, limit=50) for p in f(build_list(ex1[0]), 5)]
        ok = got == ex2[2] and got2 == ex1[2]
        print(f"{f.__name__:20s}: ví dụ 2 -> {got} | ví dụ 1 -> {got2}")
        assert not ok
    except AttributeError as e:
        print(f"{f.__name__:20s}: AttributeError ({e})")

wrong_no_cut        : ví dụ 2 -> [[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [5, 6, 7, 8, 9, 10], [8, 9, 10]] | ví dụ 1 -> [[1, 2, 3], [2, 3], [3], [], []]
wrong_extra_last    : ví dụ 2 -> [[1, 2, 3], [4, 5, 6], [7, 8, 9, 10]] | ví dụ 1 -> [[], [], [1], [2], [3]]
wrong_crash_k_gt_n  : AttributeError ('NoneType' object has no attribute 'next')
wrong_lose_rest     : ví dụ 2 -> [[1, 2, 3, 4], [], []] | ví dụ 1 -> [[1], [], [], [], []]
wrong_ceil_each     : ví dụ 2 -> [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10]] | ví dụ 1 -> [[1], [2], [3], [], []]


In [24]:
# (d) đối chiếu mọi cách với oracle chia bài, ngẫu nhiên có seed
rng = random.Random(725)
for case in range(1500):
    n = rng.choice([0, 1, 2, 3, 5, 10, 17, 50, 100, rng.randint(0, 120)])
    k = rng.choice([1, 2, 3, 7, 50, rng.randint(1, 50)])
    vals = [rng.randint(0, 1000) for _ in range(n)]
    check_all(vals, k, f"random {case}")
print("1500 input ngẫu nhiên: mọi cách khớp oracle: OK")

# vét cạn mọi cặp (n, k) nhỏ
for n in range(0, 25):
    for k in range(1, 30):
        check_all(list(range(n)), k, f"n={n}, k={k}")
print("vét cạn n = 0..24, k = 1..29: OK")

1500 input ngẫu nhiên: mọi cách khớp oracle: OK
vét cạn n = 0..24, k = 1..29: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên cắt: các đoạn dính nhau

Chỉ ghi lại đầu mỗi đoạn mà không đặt `next = None` ở nút cuối. Mọi đoạn vẫn nối tới hết danh sách gốc.

In [25]:
got = wrong_no_cut(build_list(ex2[0]), 3)
print([to_list(p) for p in got])
print("đúng:", ex2[2])

[[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], [5, 6, 7, 8, 9, 10], [8, 9, 10]]
đúng: [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]


**Vì sao sai:** danh sách liên kết chỉ "biết" điểm kết thúc qua `next = None`. Không cắt thì đoạn đầu vẫn chạy tới nút 10. **Đúng:** ở nút cuối của mỗi đoạn đặt `next = None` (sau khi lưu `next` cũ).

### Lỗi 2 — Cho phần dư các đoạn cuối

Đề yêu cầu đoạn trước dài hơn hoặc bằng đoạn sau, tức phần dư phải ưu tiên các đoạn **đầu**.

In [26]:
got = wrong_extra_last(build_list(ex2[0]), 3)
print([to_list(p) for p in got])
print("đúng:", ex2[2])

[[1, 2, 3], [4, 5, 6], [7, 8, 9, 10]]
đúng: [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]


**Vì sao sai:** `[3, 3, 4]` có đoạn trước **ngắn hơn** đoạn sau, vi phạm yêu cầu. **Đúng:** `size = q + (1 if i < r else 0)`.

### Lỗi 3 — Gọi `.next` trên `None` khi `k > n`

Khi `size = 0`, `cur` đã là `None`; viết `nxt = cur.next` (không kiểm tra) gây `AttributeError`.

In [27]:
try:
    wrong_crash_k_gt_n(build_list([1, 2, 3]), 5)
except AttributeError as e:
    print("AttributeError:", e)
print("ví dụ 2 (k <= n) vẫn đúng:", [to_list(p) for p in wrong_crash_k_gt_n(build_list(ex2[0]), 3)])

AttributeError: 'NoneType' object has no attribute 'next'
ví dụ 2 (k <= n) vẫn đúng: [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]


**Vì sao sai:** lỗi chỉ lộ khi `k > n` hoặc danh sách rỗng, nên nhiều test thường vẫn qua. **Đúng:** chỉ xử lý cắt khi `size > 0` (đoạn rỗng là `None`).

### Lỗi 4 — Cắt trước khi lưu `next`

In [28]:
got = wrong_lose_rest(build_list(ex2[0]), 3)
print([to_list(p) for p in got])
print("đúng:", ex2[2])

[[1, 2, 3, 4], [], []]
đúng: [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]


**Vì sao sai:** sau `cur.next = None` ta không còn đường tới phần còn lại; `cur.next` bây giờ là `None` nên các đoạn sau rỗng. **Đúng:** `nxt = cur.next` rồi `cur.next = None` rồi `cur = nxt`.

### Lỗi 5 — Mọi đoạn đều lấy `ceil(n / k)` nút

Nghĩ rằng "đoạn nào cũng cùng cỡ lớn nhất". Với `n = 10`, `k = 3`, `ceil = 4` nên ra `[4, 4, 2]` — hai đoạn đầu dài 4 nhưng đoạn cuối chỉ 2, chênh nhau 2.

In [29]:
got = wrong_ceil_each(build_list(ex2[0]), 3)
print([to_list(p) for p in got])
print("kích thước:", [len(to_list(p)) for p in got], "-> chênh nhau", max(map(len, map(to_list, got))) - min(map(len, map(to_list, got))))

[[1, 2, 3, 4], [5, 6, 7, 8], [9, 10]]
kích thước: [4, 4, 2] -> chênh nhau 2


**Vì sao sai:** đề cho phép chênh tối đa 1. **Đúng:** chia `n` thành `q` và `r` đoạn thêm 1 nút.

## 14. Mẹo & nhận dạng pattern

- Bài "chia/ghép/đảo/xoay danh sách liên kết" thường có hai bước: **đếm độ dài** rồi **đi đúng số bước** để xác định chỗ cắt.
- Công thức chia đều `n` cho `k`: `q, r = divmod(n, k)`; `r` phần tử đầu nhận `q + 1`. Cùng mẫu với chia kẹo, chia trang, chia tác vụ cho worker.
- Quy tắc thép khi cắt: **lưu `next` → cắt → dịch tiếp**. Viết ba dòng này liền nhau, đúng thứ tự.
- Khi `k` có thể lớn hơn `n`, hãy nghĩ ngay tới **đoạn rỗng** và kiểm tra `None` trước khi `.next`.
- Dùng `dummy` giúp xử lý "đoạn rỗng" và "nút đầu" đồng nhất (không cần `if` cho nút đầu).
- Bài trả về nhiều danh sách: test bằng cách xuất ra list Python bằng `to_list` và so sánh, đồng thời kiểm tra **không dính nhau** và **dùng lại nút gốc**.
- Nếu đề cho phép tạo nút mới, cách chép sang list rồi dựng lại là cách ít lỗi nhất để đối chiếu.

## 15. Biến thể

### 15.1. Chia theo danh sách kích thước cho trước

Cho `sizes = [2, 0, 3]` hãy cắt đúng các cỡ đó (tổng không vượt `n`; nếu thiếu nút thì đoạn sau ngắn lại).

In [30]:
def split_by_sizes(head, sizes):
    parts, cur = [], head
    for size in sizes:
        part_head = cur
        if size > 0 and cur:
            for _ in range(size - 1):
                if cur.next:
                    cur = cur.next
            nxt = cur.next
            cur.next = None
            cur = nxt
        else:
            part_head = None
        parts.append(part_head)
    return parts


print([to_list(p) for p in split_by_sizes(build_list(range(1, 8)), [2, 0, 3])])   # [[1, 2], [], [3, 4, 5]]

[[1, 2], [], [3, 4, 5]]


### 15.2. Chia đôi danh sách (đầu / cuối) bằng hai con trỏ nhanh-chậm

Ca đặc biệt `k = 2` nhưng không cần đếm: con trỏ nhanh đi 2 bước, chậm đi 1 bước; khi nhanh tới cuối thì chậm ở giữa. Nửa đầu có `ceil(n/2)` nút (đúng như bài với `k = 2`).

In [31]:
def split_half(head):
    if head is None:
        return None, None
    slow, fast = head, head.next
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    second = slow.next
    slow.next = None
    return head, second


for n in range(0, 7):
    a, b = split_half(build_list(range(n)))
    expected = [to_list(p) for p in split_count_cut(build_list(range(n)), 2)]
    assert [to_list(a), to_list(b)] == expected, n
    print(n, to_list(a), to_list(b))

0 [] []
1 [0] []
2 [0] [1]
3 [0, 1] [2]
4 [0, 1] [2, 3]
5 [0, 1, 2] [3, 4]
6 [0, 1, 2] [3, 4, 5]


### 15.3. Partition List (LeetCode 86) — chia theo giá trị, không theo độ dài

Cho `x`, đưa các nút `< x` lên trước, giữ thứ tự tương đối. Dùng hai `dummy` (nhỏ và lớn) rồi nối lại: cùng kỹ thuật "chuyển nút sang đoạn mới" như cách 4.

In [32]:
def partition(head, x):
    small, big = ListNode(), ListNode()
    s, b = small, big
    while head:
        nxt = head.next
        head.next = None
        if head.val < x:
            s.next = head
            s = head
        else:
            b.next = head
            b = head
        head = nxt
    s.next = big.next
    return small.next


print(to_list(partition(build_list([1, 4, 3, 2, 5, 2]), 3)))   # [1, 2, 2, 4, 3, 5]

[1, 2, 2, 4, 3, 5]


### 15.4. Xoay danh sách (LeetCode 61) và ghép lại các đoạn

- Ghép lại các đoạn đã chia để khôi phục danh sách gốc là phép ngược của bài này.
- Rotate List: đếm `n`, lấy `k % n`, cắt tại vị trí `n - k % n` và nối đuôi vào đầu — cũng là "đếm + cắt + lưu `next`".

In [33]:
def concat(parts):
    dummy = ListNode()
    tail = dummy
    for p in parts:
        tail.next = p
        while tail.next:
            tail = tail.next
    return dummy.next


head = build_list(range(1, 11))
parts = Solution().splitListToParts(head, 4)
print([to_list(p) for p in parts])
joined = concat(parts)          # chỉ ghép MỘT lần: ghép xong thì các đoạn đã bị nối vào nhau
print("ghép lại:", to_list(joined))
assert to_list(joined) == list(range(1, 11))

[[1, 2, 3], [4, 5, 6], [7, 8], [9, 10]]
ghép lại: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space (thêm) | Khi nào dùng |
|---|---|---|---|---|
| 1. Chép sang list | Slicing rồi dựng lại danh sách mới | `O(n + k)` | `O(n)` | Oracle kiểm tra, khi được tạo nút mới |
| 2. Đếm + cắt tại chỗ | Đi `size - 1` bước, lưu `next`, cắt | `O(n + k)` | `O(1)` | **Lời giải nộp**, tối ưu bộ nhớ |
| 3. Mảng nút + chỉ số | Công thức `start_i = i*q + min(i, r)`, cắt theo chỉ số | `O(n + k)` | `O(n)` | Khi muốn truy cập ngẫu nhiên, ít lo con trỏ |
| 4. Chuyển nút bằng `dummy` | Tháo từng nút, nối vào đoạn mới | `O(n + k)` | `O(k)` | Khi muốn code đồng nhất, tránh `.next` trên `None` |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(divmod(17, 5))
print(sizes_by_divmod(7, 3))
print([to_list(p) for p in Solution().splitListToParts(build_list([1, 2]), 4)])
```

In [34]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `n = 13` nút và `k = 5`: (a) tính `q`, `r`; (b) liệt kê kích thước từng đoạn; (c) liệt kê chỉ số bắt đầu `start_i = i*q + min(i, r)` của từng đoạn (đánh số nút từ 0).

In [35]:
# Bài 2: điền kết quả tính tay
hand_q = None
hand_r = None
hand_sizes = None      # list 5 số
hand_starts = None     # list 5 số

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo cách đếm + cắt tại chỗ.

In [36]:
def exercise_3_split(head, k):
    # TODO 1: đếm n bằng một vòng while
    # TODO 2: q, r = divmod(n, k)
    ans = []
    cur = head
    for i in range(k):
        # TODO 3: size = q + (1 nếu i < r, ngược lại 0)
        # TODO 4: ans.append(đầu đoạn)
        # TODO 5: nếu size > 0: đi size - 1 bước; lưu nxt = cur.next; cur.next = None; cur = nxt
        pass
    return ans


# assert [to_list(p) for p in exercise_3_split(build_list([1, 2, 3]), 5)] == [[1], [2], [3], [], []]
# assert [to_list(p) for p in exercise_3_split(build_list(range(1, 11)), 3)] == [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]
# assert [to_list(p) for p in exercise_3_split(None, 2)] == [[], []]

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại cách 4, hãy tự viết `exercise_4_dummy(head, k)` dùng nút `dummy` cho từng đoạn và chuyển từng nút sang.

In [37]:
def exercise_4_dummy(head, k):
    # TODO
    pass


# assert [to_list(p) for p in exercise_4_dummy(build_list([1, 2, 3]), 5)] == [[1], [2], [3], [], []]
# assert [to_list(p) for p in exercise_4_dummy(build_list(range(1, 8)), 3)] == [[1, 2, 3], [4, 5], [6, 7]]

### Bài 5 — Mini challenge: chia theo vòng tròn (khó)

Biến thể: thay vì đoạn liên tiếp, hãy **chia bài vòng tròn**: nút thứ `t` (từ 0) đưa vào đoạn `t % k`, giữ thứ tự tương đối trong mỗi đoạn. Viết `exercise_5_round_robin(head, k)` trả về `k` danh sách liên kết.

```python
[to_list(p) for p in exercise_5_round_robin(build_list(range(1, 8)), 3)]  -> [[1, 4, 7], [2, 5], [3, 6]]
[to_list(p) for p in exercise_5_round_robin(build_list([1, 2]), 4)]       -> [[1], [2], [], []]
```

In [38]:
def exercise_5_round_robin(head, k):
    # TODO
    pass


# assert [to_list(p) for p in exercise_5_round_robin(build_list(range(1, 8)), 3)] == [[1, 4, 7], [2, 5], [3, 6]]
# assert [to_list(p) for p in exercise_5_round_robin(build_list([1, 2]), 4)] == [[1], [2], [], []]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `divmod(a, b) = (a // b, a % b)`.
- Mức 2: `sizes_by_divmod(7, 3)`: `q = 2`, `r = 1`.
- Mức 3: `(3, 2)`; `[3, 2, 2]`; `[[1], [2], [], []]`.

**Bài 2**
- Mức 1: `13 // 5` và `13 % 5`.
- Mức 2: `r` đoạn đầu có `q + 1` nút.
- Mức 3: `q = 2`, `r = 3`, sizes `[3, 3, 3, 2, 2]`, starts `[0, 3, 6, 9, 11]`.

**Bài 3**
- Mức 1: Đếm giống `length`.
- Mức 2: Ghi `ans.append(cur)` **trước** khi đi bước.
- Mức 3: Xem `split_count_cut` ở mục 7.1.

**Bài 4**
- Mức 1: Mỗi đoạn: `dummy = ListNode()`, `tail = dummy`.
- Mức 2: Lặp `size` lần: `node = cur`, `cur = cur.next`, `node.next = None`, `tail.next = node`, `tail = node`.
- Mức 3: Xem `split_dummy_relink` ở mục 7.3.

**Bài 5**
- Mức 1: Giữ `k` dummy và `k` đuôi.
- Mức 2: Nút thứ `t` nối vào đoạn `t % k`.
- Mức 3: Xem đáp án mục 19.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [39]:
print(divmod(17, 5))                                                      # (3, 2)
print(sizes_by_divmod(7, 3))                                              # [3, 2, 2]
print([to_list(p) for p in Solution().splitListToParts(build_list([1, 2]), 4)])   # [[1], [2], [], []]

assert divmod(17, 5) == (3, 2)
assert sizes_by_divmod(7, 3) == [3, 2, 2]
assert [to_list(p) for p in Solution().splitListToParts(build_list([1, 2]), 4)] == [[1], [2], [], []]

(3, 2)
[3, 2, 2]
[[1], [2], [], []]


### Đáp án Bài 2

`13 // 5 = 2`, `13 % 5 = 3`. Ba đoạn đầu có 3 nút, hai đoạn cuối 2 nút: `[3, 3, 3, 2, 2]`. Chỉ số bắt đầu: `0`, `3`, `6`, `9`, `11` (`i*q + min(i, r)` với `q = 2`, `r = 3`).

In [40]:
hand_q, hand_r = 2, 3
hand_sizes = [3, 3, 3, 2, 2]
hand_starts = [0, 3, 6, 9, 11]

assert divmod(13, 5) == (hand_q, hand_r)
assert sizes_by_divmod(13, 5) == hand_sizes
assert [i * hand_q + min(i, hand_r) for i in range(5)] == hand_starts
assert hand_starts[-1] + hand_sizes[-1] == 13
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [41]:
def exercise_3_split_solution(head, k):
    n = 0
    cur = head
    while cur:
        n += 1
        cur = cur.next
    q, r = divmod(n, k)
    ans = []
    cur = head
    for i in range(k):
        size = q + (1 if i < r else 0)
        ans.append(cur)
        if size > 0:
            for _ in range(size - 1):
                cur = cur.next
            nxt = cur.next
            cur.next = None
            cur = nxt
    return ans


assert [to_list(p) for p in exercise_3_split_solution(build_list([1, 2, 3]), 5)] == [[1], [2], [3], [], []]
assert [to_list(p) for p in exercise_3_split_solution(build_list(range(1, 11)), 3)] == [[1, 2, 3, 4], [5, 6, 7], [8, 9, 10]]
assert [to_list(p) for p in exercise_3_split_solution(None, 2)] == [[], []]
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [42]:
def exercise_4_dummy_solution(head, k):
    n = 0
    cur = head
    while cur:
        n += 1
        cur = cur.next
    q, r = divmod(n, k)
    ans = []
    cur = head
    for i in range(k):
        dummy = ListNode()
        tail = dummy
        for _ in range(q + (1 if i < r else 0)):
            node = cur
            cur = cur.next
            node.next = None
            tail.next = node
            tail = node
        ans.append(dummy.next)
    return ans


assert [to_list(p) for p in exercise_4_dummy_solution(build_list([1, 2, 3]), 5)] == [[1], [2], [3], [], []]
assert [to_list(p) for p in exercise_4_dummy_solution(build_list(range(1, 8)), 3)] == [[1, 2, 3], [4, 5], [6, 7]]
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [43]:
def exercise_5_round_robin_solution(head, k):
    dummies = [ListNode() for _ in range(k)]
    tails = dummies[:]
    t = 0
    cur = head
    while cur:
        nxt = cur.next
        cur.next = None
        tails[t % k].next = cur
        tails[t % k] = cur
        cur = nxt
        t += 1
    return [d.next for d in dummies]


assert [to_list(p) for p in exercise_5_round_robin_solution(build_list(range(1, 8)), 3)] == [[1, 4, 7], [2, 5], [3, 6]]
assert [to_list(p) for p in exercise_5_round_robin_solution(build_list([1, 2]), 4)] == [[1], [2], [], []]
# kích thước khớp bài chính (chỉ khác thứ tự phần tử)
for n in range(0, 20):
    for k in range(1, 8):
        got = [len(to_list(p)) for p in exercise_5_round_robin_solution(build_list(range(n)), k)]
        assert got == sizes_by_divmod(n, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def splitListToParts(self, head: Optional[ListNode], k: int) -> List[Optional[ListNode]]:
        n = 0
        cur = head
        while cur:                       # 1. đếm độ dài
            n += 1
            cur = cur.next
        q, r = divmod(n, k)              # 2. mỗi đoạn q nút, r đoạn đầu thêm 1
        ans = [None] * k
        cur = head
        for i in range(k):
            ans[i] = cur                 # đầu đoạn (None nếu đoạn rỗng)
            size = q + (1 if i < r else 0)
            if size:
                for _ in range(size - 1):
                    cur = cur.next       # tới nút cuối đoạn
                nxt = cur.next           # 3. lưu -> cắt -> dịch
                cur.next = None
                cur = nxt
        return ans
```

### Checklist trước khi submit

- [ ] Phần dư `r` chia cho các đoạn **đầu** (`i < r`).
- [ ] Thứ tự "lưu `next` → `cur.next = None` → `cur = nxt`" đúng.
- [ ] Đoạn rỗng (`size = 0`) trả về `None` và **không** gọi `.next` trên `None`.
- [ ] Đã thử: `head = None`, `k > n`, `k = 1`, `k = n`, `n = 1000, k = 50`.
- [ ] Trả đúng `k` phần tử, đúng thứ tự xuất hiện.
- [ ] Signature đúng: `class Solution`, `splitListToParts(self, head: Optional[ListNode], k: int) -> List[Optional[ListNode]]`.